In [ ]:
import pandas as pd
patients = pd.read_csv("../data/data_clinical_patient.txt", sep="\t", skiprows=4)
patients.head()

In [ ]:
patients.shape

In [ ]:
samples = pd.read_csv("../data/data_clinical_sample.txt", sep="\t", skiprows=4)
samples.shape

In [ ]:
data = patients.merge(samples, on="PATIENT_ID", how="inner")
data.shape

In [ ]:
pd.crosstab(data["IDH_STATUS"], data["GRADE"])

In [ ]:
data["IDH_STATUS"].isna().sum()
data["GRADE"].isna().sum()
pd.crosstab(data["IDH_STATUS"].isna(), data["HISTOLOGICAL_DIAGNOSIS"])

In [ ]:
data["OS_STATUS"].value_counts()

In [ ]:
pd.crosstab(data["IDH_STATUS"], data["OS_STATUS"])

In [ ]:
data.groupby(["IDH_STATUS", "OS_STATUS"])["OS_MONTHS"].median()

In [ ]:
pd.crosstab(data["GRADE"].isna(), data["HISTOLOGICAL_DIAGNOSIS"])

In [ ]:
data[data["OS_STATUS"].isna()][["HISTOLOGICAL_DIAGNOSIS", "GRADE", "IDH_STATUS", "AGE"]].head(20)

In [ ]:
data.groupby("IDH_STATUS")["AGE"].describe()

In [ ]:
pd.crosstab(data["HISTOLOGICAL_DIAGNOSIS"], data["GRADE"])

In [ ]:
import matplotlib.pyplot as plt
data["OS_MONTHS"].hist(bins=50)
plt.xlabel("months")
plt.ylabel("patients")
plt.show()

In [ ]:
data[data["OS_STATUS"] == "1:DECEASED"]["OS_MONTHS"].hist(bins=50)
plt.title("deaths")
plt.show()

In [ ]:
data[data["OS_STATUS"] == "0:LIVING"]["OS_MONTHS"].hist(bins=50)
plt.title("censored")
plt.show()

In [ ]:
data.groupby(["IDH_STATUS", "GRADE", "OS_STATUS"])["OS_MONTHS"].median()

In [ ]:
data.groupby(["IDH_STATUS", "GRADE", "OS_STATUS"])["OS_MONTHS"].count()

In [ ]:
data[(data["IDH_STATUS"] == "Mutant") & (data["OS_STATUS"] == "0:LIVING")]["OS_MONTHS"].describe()

In [ ]:
data[(data["IDH_STATUS"] == "Mutant") & (data["OS_STATUS"] == "1:DECEASED")]["OS_MONTHS"].describe()

In [ ]:
(data["OS_MONTHS"] == 0).sum()

In [ ]:
data["event"] = (data["OS_STATUS"] == "1:DECEASED").astype(int)
km_data = data.dropna(subset=["OS_MONTHS", "OS_STATUS", "IDH_STATUS"])
km_data.shape

In [ ]:
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()
mutant = km_data[km_data["IDH_STATUS"] == "Mutant"]
kmf.fit(mutant["OS_MONTHS"], mutant["event"], label="IDH mutant")
kmf.plot_survival_function()
plt.show()

In [ ]:
fig, ax = plt.subplots()

mutant = km_data[km_data["IDH_STATUS"] == "Mutant"]
kmf_m = KaplanMeierFitter()
kmf_m.fit(mutant["OS_MONTHS"], mutant["event"], label="IDH mutant")
kmf_m.plot_survival_function(ax=ax)

wt = km_data[km_data["IDH_STATUS"] == "WT"]
kmf_w = KaplanMeierFitter()
kmf_w.fit(wt["OS_MONTHS"], wt["event"], label="IDH wildtype")
kmf_w.plot_survival_function(ax=ax)

plt.show()

In [ ]:
print(kmf_m.median_survival_time_)
print(kmf_w.median_survival_time_)

In [ ]:
from lifelines.plotting import add_at_risk_counts

fig, ax = plt.subplots(figsize=(9, 6))
kmf_m.plot_survival_function(ax=ax)
kmf_w.plot_survival_function(ax=ax)
add_at_risk_counts(kmf_m, kmf_w, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
m_median = kmf_m.median_survival_time_
w_median = kmf_w.median_survival_time_
print("mutant median:", m_median)
print("wt median:", w_median)

print("mutant at risk at its median:", kmf_m.event_table.loc[:m_median, "at_risk"].iloc[-1])
print("wt at risk at its median:", kmf_w.event_table.loc[:w_median, "at_risk"].iloc[-1])

In [ ]:
from lifelines.utils import median_survival_times
print(median_survival_times(kmf_m.confidence_interval_))
print(median_survival_times(kmf_w.confidence_interval_))

In [ ]:
from lifelines.statistics import logrank_test

result = logrank_test(mutant["OS_MONTHS"], wt["OS_MONTHS"],
                      mutant["event"], wt["event"])
result.print_summary()